# 🎭 MiniMax Face Refine (Compound Master Hybrid) - Interactive Web UI
### 2-Pass Hybrid: YOLO Face Tracking + SAM3 Semantic Inpainting + VoSR2 Super-Resolution + Gaussian Stitching
* **Architecture:** 95-node compound master pipeline chaining dynamic YOLOv8 face tracking, SAM3 semantic contour masking, 3D latent upscaling, VoSR2 wavelet super-resolution, and seamless Gaussian feathering.
* **Accelerator:** Kaggle Dual Nvidia Tesla T4 GPU (or P100)
* **Mode:** **Interactive Browser UI** (Launches ComfyUI + Public Cloudflare Web Link)


In [ ]:
# [Step 1/4] Environment & Custom Nodes Setup
import os, sys, subprocess, time

try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

t_total_start = time.time()
print('=' * 75, flush=True)
print('>>> [1/4] Installing ComfyUI & Master Hybrid Custom Nodes...', flush=True)
print('=' * 75, flush=True)

subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)

if not os.path.exists('ComfyUI'):
    subprocess.run('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git', shell=True, check=True)

custom_nodes = 'ComfyUI/custom_nodes'
os.makedirs(custom_nodes, exist_ok=True)

repos = [
    'https://github.com/ltdrdata/ComfyUI-Manager.git',
    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'https://github.com/fearnworks/comfyui_fearnworksnodes.git',
    'https://github.com/kijai/ComfyUI-KJNodes.git',
    'https://github.com/drozbay/MaskVidExperiments.git'
]

for repo in repos:
    target = os.path.join(custom_nodes, os.path.splitext(os.path.basename(repo))[0])
    if not os.path.exists(target):
        subprocess.run(f'git clone --depth 1 {repo} {target}', shell=True, check=True)

pkgs = ['huggingface_hub>=0.28.0', 'hf_transfer', 'opencv-python-headless', 'ultralytics', 'imageio', 'imageio-ffmpeg', 'pillow', 'requests', 'tqdm']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ComfyUI/requirements.txt'], check=True)
print('✓ Environment and custom node packages ready.', flush=True)


In [ ]:
# [Step 2/4] High-Speed Model Provisioning (Mounted Dataset + HF Fallback)
import os, shutil, glob, time
from huggingface_hub import hf_hub_download

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

print('=' * 75, flush=True)
print('>>> [2/4] Provisioning Model Checkpoints...', flush=True)
print('=' * 75, flush=True)

os.makedirs('ComfyUI/models/diffusion_models', exist_ok=True)
os.makedirs('ComfyUI/models/text_encoders', exist_ok=True)
os.makedirs('ComfyUI/models/vae', exist_ok=True)
os.makedirs('ComfyUI/models/latent_upscalers', exist_ok=True)
os.makedirs('ComfyUI/models/upscale_models', exist_ok=True)
os.makedirs('ComfyUI/models/ultralytics/bbox', exist_ok=True)
os.makedirs('ComfyUI/models/checkpoints', exist_ok=True)

# Symlink text_encoders to clip so stock loaders resolve
if not os.path.exists('ComfyUI/models/clip'):
    try: os.symlink('text_encoders', 'ComfyUI/models/clip')
    except Exception: pass

ds_candidates = glob.glob('/kaggle/input/*minimax*weights*')
ds_root = ds_candidates[0] if ds_candidates else '/kaggle/input/minimax-h3-open-weights'
print(f'Detected Mounted Dataset Root: {ds_root} (Exists: {os.path.exists(ds_root)})', flush=True)

def link_or_fetch(rel_ds, dst, hf_repo, hf_file):
    src = os.path.join(ds_root, rel_ds)
    if os.path.exists(src):
        if not os.path.exists(dst):
            try: os.symlink(src, dst)
            except Exception: shutil.copyfile(src, dst)
        print(f'✓ [MOUNTED 0-SEC] {dst}', flush=True)
    else:
        if not os.path.exists(dst):
            print(f'⏳ Downloading from HF: {hf_file}...', flush=True)
            p = hf_hub_download(repo_id=hf_repo, filename=hf_file)
            try: os.symlink(p, dst)
            except Exception: shutil.copyfile(p, dst)
            print(f'✓ [DOWNLOADED] {dst}', flush=True)
        else:
            print(f'✓ [CACHED] {dst}', flush=True)

# 1. Base MiniMax weights
link_or_fetch('clip/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'ComfyUI/models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors')
link_or_fetch('vae/minimax_h3_video_vae_int8_convrot.safetensors', 'ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors', 'Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors')
link_or_fetch('vae/minimax_h3_audio_vae_fp32.safetensors', 'ComfyUI/models/vae/minimax_h3_audio_vae_fp32.safetensors', 'Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors')
link_or_fetch('diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', 'ComfyUI/models/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', 'Comfy-Org/MiniMax-H3', 'diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors')

# 2. 3D Latent Upscaler & VoSR2
link_or_fetch('latent_upscalers/minimax_h3_latent_upscaler_3d_bf16.safetensors', 'ComfyUI/models/latent_upscalers/minimax_h3_latent_upscaler_3d_bf16.safetensors', 'fearnworks/vosr2', 'minimax_h3_latent_upscaler_3d_bf16.safetensors')
link_or_fetch('upscale_models/vosr2_wavelet_fp16.safetensors', 'ComfyUI/models/upscale_models/vosr2_wavelet_fp16.safetensors', 'fearnworks/vosr2', 'vosr2_wavelet_fp16.safetensors')

# 3. YOLO Face Detector & SAM3
yolo_dst = 'ComfyUI/models/ultralytics/bbox/face_yolov8m.pt'
if not os.path.exists(yolo_dst):
    subprocess.run(f'wget -q -c https://github.com/akanametov/yolo-face/releases/download/v0.0.0/face_yolov8m.pt -O {yolo_dst}', shell=True)
    print('✓ [DOWNLOADED] face_yolov8m.pt', flush=True)

sam_dst = 'ComfyUI/models/checkpoints/sam3.1_multiplex_fp16.safetensors'
link_or_fetch('checkpoints/sam3.1_multiplex_fp16.safetensors', sam_dst, 'fearnworks/vosr2', 'sam3.1_multiplex_fp16.safetensors')

print('✓ All models provisioned and verified.', flush=True)


In [ ]:
# [Step 3/4] Ingest Workflows & Demo References
import os, shutil

wf_dir = 'ComfyUI/user/default/workflows'
input_dir = 'ComfyUI/input'
os.makedirs(wf_dir, exist_ok=True)
os.makedirs(input_dir, exist_ok=True)

local_wf = 'workflows'
if os.path.exists(local_wf):
    for f in os.listdir(local_wf):
        src_p = os.path.join(local_wf, f)
        if f.endswith('.json'):
            shutil.copy(src_p, wf_dir)
        elif f.lower().endswith(('.jpg', '.png', '.mp4')):
            shutil.copy(src_p, input_dir)

print(f'✓ Deployed Master Hybrid workflow: minimax_h3_r2v_perfect_face_refine.json into ComfyUI Workflows menu!', flush=True)
print('✓ Reference demo assets staged into ComfyUI/input/', flush=True)


In [ ]:
# [Step 4/4] Launch Interactive ComfyUI Server & Cloudflare Live Link
import subprocess, threading, time, re

PORT = 8188
print('=' * 75, flush=True)
print('>>> [4/4] Starting ComfyUI Server & Cloudflare Tunnel...', flush=True)
print('=' * 75, flush=True)

# Launch with lowvram to allow clean sequential offloading of 32B text encoder and 14B DiT
comfy_proc = subprocess.Popen(
    f'{sys.executable} ComfyUI/main.py --listen 127.0.0.1 --port {PORT} --enable-cors-header --lowvram --reserve-vram 1.5',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

def monitor():
    for line in comfy_proc.stdout:
        if any(w in line for w in ['Loaded', 'Prompt', 'Executing', 'error', 'Error', 'SelfLift', 'VOSR2', 'SAM3']):
            print(f'[ComfyUI] {line.strip()}', flush=True)

threading.Thread(target=monitor, daemon=True).start()

tunnel_proc = subprocess.Popen(f'/usr/local/bin/cloudflared tunnel --url http://127.0.0.1:{PORT}', shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public_url = None
t_start = time.time()
while time.time() - t_start < 45:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5); continue
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0); break

print('\n' + '=' * 80, flush=True)
if public_url:
    print(f'🌐 COMFYUI LIVE WEB UI: {public_url}', flush=True)
    print('👉 Open the link above, click Workflows menu, and select minimax_h3_r2v_perfect_face_refine.json!', flush=True)
print('=' * 80 + '\n', flush=True)

counter = 0
while True:
    time.sleep(60)
    counter += 1
    if counter % 5 == 0:
        print(f'[Heartbeat] Master Hybrid Face Refine server is online ({counter} min).', flush=True)
